# Getting started

We make E- and B-mode convergence maps from the shear catalog of a simulated galaxy cluster, compute their
signal-to-noise maps and write them to FITS. We then repeat the run from a YAML configuration file, as the `smpy`
command does.

The catalog is a simulated SuperBIT observation of a cluster (McCleary et al. 2023), with a shear estimate and a
weight for each galaxy.

## Settings

These are the values to change for another catalog: its path, the names of its columns and the size of the map
pixels in arcminutes. Paths are relative to `docs/source/tutorials` in the SMPy repository.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS

import smpy

CATALOG = Path("../../../examples/data/forecast_lum_annular.fits")
COLUMNS = dict(coord1_col="ra", coord2_col="dec", g1_col="g1_Rinv", g2_col="g2_Rinv", weight_col="weight")
PIXEL_SCALE = 0.4  # arcminutes

## The catalog

SMPy reads a FITS table, or an astropy `Table`, with a position, two shear components and optionally a weight for
each galaxy.

In [ ]:
catalog = Table.read(CATALOG, hdu=1)
print(f"{len(catalog)} galaxies")
catalog[list(COLUMNS.values())][:5]

## Mass maps

`smpy.map_mass` bins the shear into pixels and inverts it into convergence, by default with the Kaiser-Squires
inversion (Kaiser & Squires 1993) and a Gaussian smoothing of 2 pixels. It returns a `MassMap` with the E- and
B-mode maps, the binned catalog and every setting used.

In [ ]:
result = smpy.map_mass(CATALOG, pixel_scale=PIXEL_SCALE, **COLUMNS)
result.maps["E"].shape

Gravitational lensing produces only E modes. The cluster therefore appears in the E-mode map, while the B-mode map
should look like noise; structure in it points to systematic errors in the shear.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"wspace": 0.4})
result.plot("E", ax=axes[0], cmap="magma")
result.plot("B", ax=axes[1], cmap="magma")
plt.show()

## Signal-to-noise maps

With `create_snr=True`, SMPy also maps 100 copies of the catalog in which every shear is turned by a random
angle. This removes the lensing signal and keeps the shape noise, and the signal-to-noise (SNR) map is the map
divided by the standard deviation of these maps in each pixel. The signal-to-noise tutorial covers this in detail.

In [ ]:
result = smpy.map_mass(CATALOG, pixel_scale=PIXEL_SCALE, create_snr=True, **COLUMNS)
result.plot("E", snr=True, cmap="magma")
plt.show()
print(f"Peak E-mode SNR: {result.snr['E'].max():.1f}")

## FITS output

`to_fits` writes a map with a world coordinate system (WCS) for its tangent-plane projection, so other software
places every pixel on the sky. Here we read the position of the highest pixel back from the file.

In [ ]:
output = TemporaryDirectory()
path = Path(output.name) / "convergence_e.fits"
result.to_fits(path, "E")

with fits.open(path) as hdus:
    data, wcs = hdus[0].data, WCS(hdus[0].header)
y, x = np.unravel_index(np.argmax(data), data.shape)
wcs.pixel_to_world(x, y)

## Running from a configuration file

A YAML file can describe the same run. `smpy.run` reads it, makes the maps and writes plots, FITS files and a record
of every setting used; `smpy -c config.yaml` does the same from the command line. Settings that the file leaves out
keep the defaults listed on the configuration page.

In [ ]:
config_path = Path(output.name) / "config.yaml"
config_path.write_text(f'''
general:
  input_path: {CATALOG.resolve()}
  output_directory: {output.name}
  radec:
    pixel_scale: {PIXEL_SCALE}
  g1_col: {COLUMNS["g1_col"]}
  g2_col: {COLUMNS["g2_col"]}
  weight_col: {COLUMNS["weight_col"]}
  mode: [E, B]
  create_snr: true
  save_fits: true
''')
run_result = smpy.run(config_path)
sorted(path.name for path in (Path(output.name) / "kaiser_squires").iterdir())

The settings record lists every setting the run used and the SMPy version. Passing it back to `smpy.run` repeats the
run exactly.

In [ ]:
record = Path(output.name) / "kaiser_squires" / "smpy_output_kaiser_squires_settings.yaml"
print("\n".join(record.read_text().splitlines()[:20]))

repeat = smpy.run(record)
print("Identical maps:", all(np.array_equal(run_result.snr[mode], repeat.snr[mode]) for mode in "EB"))